In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

In [2]:
pio.renderers.default = 'iframe'
import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
import gensim; gensim.__version__

'4.4.0'

In [3]:
corpus_file = "/kaggle/input/notebooks/isabellouisedelgado/text-as-data-final-corpus/GothicNovels_CORPUS.csv"
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
BAG = OHCO[:4]

In [4]:
TOKENS = pd.read_csv(corpus_file).set_index(OHCO).dropna()
TOKENS.head()

token_str  pos pos_group  \
book_id chap_num para_num sent_num token_num                            
696     1        1        0        0           Manfred  NNP        NN   
                                   2            Prince  NNP        NN   
                                   3                of   IN        IN   
                                   4           Otranto  NNP        NN   
                                   6               had  VBD        VB   

                                             term_str        period  
book_id chap_num para_num sent_num token_num                         
696     1        1        0        0          manfred  early gothic  
                                   2           prince  early gothic  
                                   3               of  early gothic  
                                   4          otranto  early gothic  
                                   6              had  early gothic

In [5]:
lib_source = "/kaggle/input/notebooks/isabellouisedelgado/text-as-data-final-corpus/GothicNovels_LIB.csv"
LIB = pd.read_csv(lib_source).set_index('book_id')
LIB.tail()

,source_file_path,author,title,chap_pat,clip_pats,period,book_len,n_chaps
book_id,,,,,,,,
6087,/kaggle/input/datasets/isabellouisedelgado/cla...,"POLIDORI, JOHN",THE VAMPYRE,^\s*THE VAMPYRE\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic,10991,2
12122,/kaggle/input/datasets/isabellouisedelgado/mod...,"JACOBS, WW",THE MONKEYS PAW,^\s*[IVXLCM]+\.\s*$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",modern gothic,4023,3
41445,/kaggle/input/datasets/isabellouisedelgado/cla...,"SHELLEY, MARY",FRANKENSTEIN,^(?:LETTER|CHAPTER)\s+[IVXLCM]+\.$,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic,72069,27
53685,/kaggle/input/datasets/isabellouisedelgado/cla...,"MATURIN, CHARLES",MELMOTH THE WANDERER,^(?:CHAPTER\s+[IVXLCM]+\.|Tale of the Spaniard...,"['\\*\\*\\*\\s*START OF', '\\*\\*\\*\\s*END OF']",classical gothic,57022,6
20180856,/kaggle/input/datasets/isabellouisedelgado/mod...,"JACKSON, SHIRLEY",THE HAUNTING OF HILL HOUSE,^(?:ONE|TWO|THREE|FOUR|FIVE|SIX|SEVEN|EIGHT|NI...,"['_For Leonard Brown_', 'TRANSCRIBER NOTES']",modern gothic,65454,9


In [6]:
DOCS = TOKENS.join(LIB.author).groupby(['author']+BAG).term_str.apply(list)
DOCS.head()

author             book_id  chap_num  para_num  sent_num
GILMAN, CHARLOTTE  1952     1         0         0                            [by, charlotte, perkins, gilman]
                                      1         0           [it, is, very, seldom, that, mere, ordinary, p...
                                      2         0           [a, colonial, mansion, a, hereditary, estate, ...
                                      3         0           [still, i, will, proudly, declare, that, there...
                                      4         0               [else, why, should, it, be, let, so, cheaply]
Name: term_str, dtype: object

In [7]:
w2v_params = dict(
    window = 2,
    vector_size = 256,
    min_count = 50,
    workers = 1,
    seed = 42
)

In [8]:
model = word2vec.Word2Vec(DOCS, **w2v_params)
model.wv.vectors

array([[ 0.08100431,  0.14849281,  0.28988633, ..., -0.18243279,
         0.01844676, -0.10969886],
       [ 0.16970831, -0.1170811 , -0.10486193, ..., -0.3030996 ,
         0.35292047,  0.01801456],
       [ 0.44940585, -0.03401021,  0.35097688, ...,  0.22838755,
         0.09533118,  0.15542723],
       ...,
       [ 0.11334144, -0.07783394,  0.04776036, ..., -0.07386668,
        -0.01604566,  0.01321722],
       [ 0.10050995, -0.0853677 ,  0.10277665, ...,  0.02389993,
        -0.03701067,  0.07176821],
       [ 0.0584037 , -0.1224527 ,  0.03724609, ..., -0.10846107,
        -0.0765411 ,  0.03498272]], dtype=float32)

## VOCAB_W2V (4)
A table of word2vec features associated with terms in the VOCAB table.

In [9]:
WV = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
WV.index.name = 'term_str'
WV

,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
term_str,,,,,,,,,,,,,,,,,,,,,
the,0.081004,0.148493,0.289886,-0.202663,-0.251074,0.334240,0.012874,0.043751,-0.470161,-0.219275,...,0.085178,0.227291,-0.125594,-0.352007,-0.048836,-0.473125,-0.105073,-0.182433,0.018447,-0.109699
and,0.169708,-0.117081,-0.104862,0.295275,0.259087,0.104036,-0.275871,0.446255,0.053072,0.092286,...,0.243073,0.208529,-0.172189,0.121680,-0.007683,-0.042529,-0.301913,-0.303100,0.352920,0.018015
to,0.449406,-0.034010,0.350977,-0.174059,0.832425,-0.635624,-0.433314,-0.108447,-0.299121,0.208953,...,-0.553643,-0.637062,-0.099189,0.124294,0.534835,0.040085,0.288119,0.228388,0.095331,0.155427
of,-0.611498,-0.148469,-0.312317,0.588250,-0.314958,0.272721,0.248356,0.692641,-0.557608,-0.608042,...,-0.601187,0.059478,0.046967,-0.420173,-0.165317,-0.560633,-0.316900,-0.226703,-0.401176,0.292381
i,0.362989,-0.461379,-0.321629,-0.302036,0.324871,-0.682974,0.234677,-0.783378,0.112956,-0.114225,...,0.001500,-0.856996,0.362217,0.553120,0.601976,0.200291,-0.173419,0.195579,-0.380709,0.617268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
weeks,0.102184,-0.160272,-0.093231,0.154428,-0.033496,-0.063133,-0.207387,-0.090301,-0.231824,-0.133128,...,-0.225314,0.094490,-0.158749,0.055362,-0.057154,-0.205246,-0.134572,-0.122276,0.015591,-0.066767
bit,0.050643,-0.098708,0.019786,-0.030484,-0.095173,-0.031782,-0.247850,0.110296,-0.084248,-0.106968,...,-0.031366,0.079492,0.040706,0.052595,-0.042998,-0.106036,-0.124005,-0.054479,0.014016,-0.010472
sick,0.113341,-0.077834,0.047760,-0.028855,0.001071,-0.030841,-0.074274,0.068617,-0.110733,-0.081743,...,-0.136983,0.103336,0.008718,0.006361,0.004065,-0.188231,-0.059976,-0.073867,-0.016046,0.013217


In [10]:
tsne_params = dict(
    perplexity=50, 
    n_components=2, 
    init='pca', 
    max_iter=1000, 
    random_state=42    
)

In [11]:
tsne_engine = tsne(**tsne_params)

TSNE = pd.DataFrame(
    tsne_engine.fit_transform(WV), 
    columns=['x','y'], 
    index=WV.index)

TSNE

,x,y
term_str,,
the,27.824776,-0.740142
and,4.527518,9.752913
to,-15.135011,8.854500
of,1.048674,-2.547271
i,-20.987480,8.516495
...,...,...
weeks,20.777605,-10.743269
bit,-3.177102,-2.815134
sick,-5.256995,1.154931


In [12]:
px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')